# Imports/Settings

In [2]:
%load_ext autoreload
%autoreload 2

In [15]:
# 1. Стандартная библиотека
import os
import sys
import warnings
from pathlib import Path

# --- ДИНАМИЧЕСКИЙ РАСЧЕТ АБСОЛЮТНЫХ ПУТЕЙ ---
notebook_dir = Path(os.getcwd()).resolve()
if notebook_dir.name == "notebooks":
    PROJECT_ROOT = notebook_dir.parent
else:
    PROJECT_ROOT = notebook_dir

os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))


# 2. Сторонние библиотеки
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
import sweetviz as sv
from omegaconf import OmegaConf

# 3. Локальные модули
from core.data import UniversalDataLoader
from core.features import TabularPreprocessor, FeatureEngineer
from core.utils import load_hydra_config, collapse_rare_categories

# --- РАБОТА С ПРЕДУПРЕЖДЕНИЯМИ ---
warnings.filterwarnings('ignore', category=FutureWarning)

# --- ИНИЦИАЛИЗАЦИЯ HYDRA ---
load_hydra_config.cache_clear()
# --- Инициализация Гидры ---
cfg = load_hydra_config()

print(f"Проект: {cfg.project_name} | Режим: Feature Engineering")
print(f"Проверка sample_pct: {cfg.data.sample_pct}")


reports_dir = Path(cfg.paths.reports_dir)
reports_dir.mkdir(parents=True, exist_ok=True)

# --- НАСТРОЙКИ ВИЗУАЛИЗАЦИИ ---
try:
    p_cfg = cfg.logging.plots
    plt.style.use(p_cfg.style)
    plt.rcParams.update({
        'figure.figsize': p_cfg.fig_size,
        'figure.dpi': p_cfg.dpi,
        'font.size': p_cfg.font_size,
        'axes.grid': p_cfg.grid,
        'axes.spines.top': p_cfg.spines_top,
        'axes.spines.right': p_cfg.spines_right
    })
    PLOT_ALPHA = p_cfg.alpha
except AttributeError:
    PLOT_ALPHA = 0.5
    print("Используются дефолтные стили Matplotlib.")

Проект: outsource_project_name | Режим: Feature Engineering
Проверка sample_pct: 0.05


# Data Loading

In [4]:
loader = UniversalDataLoader(cfg)

df = loader.load_data()
target = cfg.data.tabular.target_col

print(f"Размер датасета: {df.shape}")
df.head(3)

Размер датасета: (92804, 30)


,session_id,client_id,visit_date,visit_time,visit_number,utm_source,utm_medium,utm_campaign,utm_adcontent,utm_keyword,...,first_hit_time_ms,last_hit_time_ms,total_events_count,unique_event_actions,unique_cars_viewed,total_car_views,is_first_hit_car_view,event_value,hits_before_target,car_view_ratio
0,1000066343190736009.1639220491.1639220491,232846090.16392636,2021-12-11,14:01:31,2,kjsLglQLzykiRbcDiGcD,cpc,None,None,nSReTmyFtbSjlPrTKoaX,...,9936.0,29597.0,2,2,0,0,0,0,2,0.0000
1,1000221619142178119.1638053194.1638053194,232882243.1638053191,2021-11-28,01:46:34,1,ZpYIoDJMcFzVoPFsHGJL,banner,LEoPHuyFvzoNfnzGgfcd,JNHcPlZPxEMWDnRiyoBf,puhZPIYqKXeFPaUviSjo,...,231.0,231.0,1,1,0,0,0,0,1,0.0000
2,1000470950571588275.1625995955.1625995955,232940295.1625995955,2021-07-11,12:00:00,1,ZpYIoDJMcFzVoPFsHGJL,banner,LEoPHuyFvzoNfnzGgfcd,JNHcPlZPxEMWDnRiyoBf,None,...,0.0,0.0,7,5,1,5,0,0,7,0.7143


In [4]:
df.to_parquet(f"{cfg.paths.features_dir}/df_aggregated.pq", index=False)

In [5]:
target_col = cfg.data.tabular.target_col # 'event_value'
y_train = df[target_col].values
X_train_raw = df.drop(columns=[target_col])

# Feature Generation

In [6]:
tabular_cfg = cfg.get('data', {}).get('tabular', {})
geo_cfg = tabular_cfg.get('geo', {})
devices_cfg = tabular_cfg.get('devices', {})

cis_countries = list(geo_cfg.get('cis', []))
mobile_cats = list(devices_cfg.get('mobile_categories', ['mobile', 'tablet']))

# 2. Извлекаем справочник городов прямо из пути cfg.data.tabular
# Конвертируем в нативный dict для максимальной скорости работы .map() в Pandas
city_markets_cfg = tabular_cfg.get('city_markets', {})
city_markets = OmegaConf.to_container(city_markets_cfg, resolve=True) if city_markets_cfg else {}

# Загружаем дефолтный профиль на случай редких/пропущенных локаций
defaults_cfg = tabular_cfg.get('defaults_fallback', {})
city_defaults = OmegaConf.to_container(defaults_cfg, resolve=True) if defaults_cfg else {
    'has_metro': 0, 'population_2021': 100000, 'avg_salary_2021': 33000, 'cars_per_family': 0.65
}


In [7]:
def transform(X: pd.DataFrame) -> pd.DataFrame:
        X_transformed = X.copy()

        # ==========================================================
        # 1. СИНХРОНИЗАЦИЯ ЗАГЛУШЕК (Убираем конфликт Other)
        # ==========================================================
        for col in X_transformed.select_dtypes(include=['object']).columns:
            X_transformed[col] = X_transformed[col].replace(['(Other)', '(not set)', 'other'], np.nan)

        # ==========================================================
        # 2. ПАРСИНГ ПЛОЩАДИ ЭКРАНА
        # ==========================================================
        if 'device_screen_resolution' in X_transformed.columns:
            res_split = X_transformed['device_screen_resolution'].astype(str).str.split('x', expand=True)
            if res_split.shape[1] >= 2:
                width = pd.to_numeric(res_split[0], errors='coerce')
                height = pd.to_numeric(res_split[1], errors='coerce')
                X_transformed['screen_area'] = width * height
            else:
                X_transformed['screen_area'] = np.nan
        else:
            X_transformed['screen_area'] = np.nan

        # ==========================================================
        # 3. ГЕО-ЗОНЫ И МОБИЛЬНОСТЬ
        # ==========================================================
        if 'geo_country' in X_transformed.columns:
            conditions = [
                X_transformed['geo_country'] == 'Russia',
                X_transformed['geo_country'].isin(cis_countries)
            ]
            choices = ['russia', 'cis']
            X_transformed['geo_zone'] = np.select(conditions, choices, default='other_countries')
        else:
            X_transformed['geo_zone'] = 'other_countries'

        if 'device_category' in X_transformed.columns:
            X_transformed['is_mobile_device'] = X_transformed['device_category'].isin(mobile_cats).astype(int)
        else:
            X_transformed['is_mobile_device'] = 0

        # ==========================================================
        # 4. КАЛЕНДАРНЫЕ И ВРЕМЕННЫЕ Признаки
        # ==========================================================
        if 'visit_date' in X_transformed.columns:
            date_series = pd.to_datetime(X_transformed['visit_date'], errors='coerce')
            X_transformed['day_of_week'] = date_series.dt.dayofweek.fillna(0).astype(int)
            X_transformed['is_weekend'] = X_transformed['day_of_week'].isin([5, 6]).astype(int)

        if 'visit_time' in X_transformed.columns:
            hours = pd.to_datetime(X_transformed['visit_time'], format='%H:%M:%S', errors='coerce').dt.hour
            if hours.isnull().all():
                hours = pd.to_numeric(X_transformed['visit_time'], errors='coerce').fillna(12)
            X_transformed['is_night'] = hours.isin([23, 0, 1, 2, 3, 4, 5]).astype(int)

        # ==========================================================
        # 5. МАТЕМАТИКА СКОРОСТИ КЛИКОВ И КРОССЫ
        # ==========================================================
        if 'last_hit_time_ms' in X_transformed.columns and 'first_hit_time_ms' in X_transformed.columns:
            X_transformed['session_duration_ms'] = X_transformed['last_hit_time_ms'] - X_transformed['first_hit_time_ms']
            if 'total_hits_count' in X_transformed.columns:
                X_transformed['ms_per_hit'] = np.where(
                    X_transformed['total_hits_count'] > 0,
                    X_transformed['session_duration_ms'] / X_transformed['total_hits_count'],
                    0
                )

        if 'device_category' in X_transformed.columns and 'device_brand' in X_transformed.columns:
            X_transformed['dev_category_brand'] = (
                X_transformed['device_category'].astype(str) + "_" + X_transformed['device_brand'].astype(str)
            )

        # ==========================================================
        # 6. ВСТРОЕННОЕ ОБОГАЩЕНИЕ ДЕМОГРАФИЕЙ ГОРОДОВ (Из конфига)
        # ==========================================================
        if 'geo_city' in X_transformed.columns:
            # Нормализуем строку города для точного мэтчинга с ключами YAML
            city_normalized = X_transformed['geo_city'].astype(str).str.lower().str.strip()
            
            # Маппим признаки, используя кэшированный нативный python-dict
            X_transformed['has_metro'] = city_normalized.map(
                lambda x: city_markets.get(x, {}).get('has_metro', city_defaults['has_metro'])
            )
            X_transformed['city_population'] = city_normalized.map(
                lambda x: city_markets.get(x, {}).get('population_2021', city_defaults['population_2021'])
            )
            X_transformed['city_avg_salary'] = city_normalized.map(
                lambda x: city_markets.get(x, {}).get('avg_salary_2021', city_defaults['avg_salary_2021'])
            )
            X_transformed['city_cars_per_family'] = city_normalized.map(
                lambda x: city_markets.get(x, {}).get('cars_per_family', city_defaults['cars_per_family'])
            )
        else:
            # Если колонки нет, заполняем дефолтными значениями
            X_transformed['has_metro'] = city_defaults['has_metro']
            X_transformed['city_population'] = city_defaults['population_2021']
            X_transformed['city_avg_salary'] = city_defaults['avg_salary_2021']
            X_transformed['city_cars_per_family'] = city_defaults['cars_per_family']

        # 7. Фича-пропорция (интерес юзера к авто относительно среднего по городу)
        if 'total_car_views' in X_transformed.columns:
            X_transformed['user_vs_city_car_interest'] = X_transformed['total_car_views'] / (X_transformed['city_cars_per_family'] + 1e-5)

        return X_transformed

In [8]:
X_train = transform(X_train_raw)

In [12]:
preprocessor = TabularPreprocessor(cfg)
X_train_processed = preprocessor.fit_transform(X_train)

In [16]:
X_train_encoded = X_train_processed.copy()
cat_cols = X_train_encoded.select_dtypes(include=['object']).columns.tolist()
if cat_cols:
    encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    X_train_encoded[cat_cols] = encoder.fit_transform(X_train_encoded[cat_cols].astype(str))

In [22]:
mi_scores = mutual_info_classif(X_train_encoded, y_train)
mi_df = pd.DataFrame({'Feature': X_train_encoded.columns, 'MI': mi_scores})
print(mi_df.sort_values(by='MI', ascending=False))

                      Feature        MI
16       unique_event_actions  0.027409
22                   geo_zone  0.023055
18            total_car_views  0.022329
34  user_vs_city_car_interest  0.022149
12           total_hits_count  0.021664
15         total_events_count  0.021111
10                geo_country  0.018660
6             device_category  0.018327
23           is_mobile_device  0.017867
30                  has_metro  0.017524
20             car_view_ratio  0.016754
17         unique_cars_viewed  0.013276
5                 utm_keyword  0.011159
31            city_population  0.009603
7                   device_os  0.006978
3                utm_campaign  0.006955
32            city_avg_salary  0.006173
14           last_hit_time_ms  0.006011
33       city_cars_per_family  0.005570
9              device_browser  0.005258
2                  utm_medium  0.005199
11                   geo_city  0.005198
1                  utm_source  0.005098
8                device_brand  0.004203


In [23]:
eda_df = X_train_encoded.copy()
eda_df[target_col] = y_train

# Схлопываем редкие категории, чтобы Sweetviz построил красивые графики без каши
cat_cols = eda_df.select_dtypes(exclude=[np.number]).columns.tolist()
eda_df_collapsed = collapse_rare_categories(eda_df, cat_cols, top_n=12)

feature_config = sv.FeatureConfig()
feature_config.force_num = [target_col]
feature_config.force_cat = ['day_of_week', 'is_weekend', 'is_night']

report = sv.analyze(eda_df_collapsed, target_feat=target_col, feat_cfg=feature_config)
output_path = reports_dir / "eda_featuredv2.0.html"
report.show_html(filepath=str(output_path), open_browser=True)

                                             |          | [  0%]   00:00 -> (? left)

Report reports\eda_featuredv2.0.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


# Выводы по Feature Engineering

## Какие категории фич были добавлены
- Парсинг площади экрана из разрешения
- Глобальная региональность (Россия, страны СНГ, остальной мир)
- календарные и временные фичи (день,месяц, день недели, время суток)
- математика вокруг кликов (время между кликами, количество кликов)
- комбинирование категории девайса и бренда
- внешние фичи связанные с демографией и статистикой городов

## Идеи для новых фичей
- таргет энкодер по различным фичам таким как(категория девайса, код машины(марка+модель), geo_zone, geo_city) Однако я буду использовать catboost т.к. очень много категориальных фич, поэтому это действие не нужно
- аггрегирование математических статистик по фичам таким как (модель телефона, операционная система, модель автомобиля) таких фичей как (среднее количество кликов, screen_area, потенциально любых числовых фичей)
- попробовать соединить город и код машины т.к. вероятно есть связь между классом машины и городом (допустим внедорожник или машина представительского класса)
- поведенческие сдвиги (такие как total_hits_count пользователя / (медианное количество кликов для его device_brand))
- динамика внутри сессии (например is_fast_session если среднее между кликами меньше 500мс то это бот и у него конверсии не будет)

## Влияние фич предварительное (по mi_score)

| Feature | MI |
| :--- | :--- |
| unique_event_actions | 0.027409 |
| geo_zone | 0.023055 |
| total_car_views | 0.022329 |
| user_vs_city_car_interest | 0.022149 |
| total_hits_count | 0.021664 |
| total_events_count | 0.021111 |
| geo_country | 0.018660 |
| device_category | 0.018327 |
| is_mobile_device | 0.017867 |
| has_metro | 0.017524 |
| car_view_ratio | 0.016754 |
| unique_cars_viewed | 0.013276 |
| utm_keyword | 0.011159 |


Новая фича user_vs_city_car_interest, созданная на стыке логов и демографии городов, показала высокий MI_Score (0.022149), заняв 4-е место в общем рейтинге. Это доказывает успешность гипотезы об обогащении данных внешними макро-показателями рынков.

## Предварительно бесполезные фичи (по mi_score)

| Feature | MI |
| :--- | :--- |
| first_hit_time_ms | 0.000000 |
| is_first_hit_car_view | 0.000000 |